# Fit a Born machine to bitstrings

Train a three-qubit model on a distribution supported on 000 and 111. Read exact model probabilities, generate finite samples, and draw the sampling circuit.

Run the cells in order with QASP 0.1.2. See the [tutorial setup guide](../README.md) for installation and optional PDF rendering.

In [ ]:
%matplotlib inline
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)

## Configure the experiment

The fixed conditioning input is encoded by a scaled AngleEmbedding. The training data are target bitstrings, whereas inputs sets the angles of the encoding circuit. An unconditional Born machine can omit the feature map, inputs and input_scaling_init.

In [ ]:
from qasp import Normal, Uniform
from qasp import create_qcbm, draw_qcbm

n_qubits = 3
data = np.array([[0, 0, 0], [1, 1, 1]] * 8, dtype=int)
config = {
    "init": Normal(0, 0.4), "input_scaling_init": Uniform(0.8, 1.2),
    "feature_map": "AngleEmbedding",
    "feature_map_config": {"features": 3, "rotations": "Y", "input_scaling": True},
    "inputs": [0.25, -0.4, 0.6],
    "ansatz": "AngleAnsatz",
    "ansatz_config": {
        "n_layers": 2, "rotations": "YZ",
        "include_imprimitive": True, "imprimitive": "CNOT",
        "imprimitive_pattern": "chain",
    },
    "loss": "mmd",
    "mmd_config": {"mode": "exact", "bandwidths": [0.5, 1.0, 2.0]},
    "interface": "torch", "diff_method": "backprop",
    "device": "default.qubit", "device_kwargs": {"seed": 7},
    "shots": None, "seed": 7, "lr": 0.05,
}

## Construct and inspect the initial model

Run this cell again after editing the configuration to sample fresh parameters.

In [ ]:
model = create_qcbm(config, n_qubits=n_qubits, data=data)
print("Trainable tensor shapes:", [tuple(p.shape) for p in model.parameters()])
print("Initial angles:", model.params.detach())
print("Initial encoding weights:", model.input_scaling_params.detach())
initial_probs = model.get_probs().detach().clone()

## Train

The MMD loss compares the generated and target distributions over several bandwidths. Exact mode makes the short loss trajectory reproducible with the supplied seed.

In [ ]:
trained_params, history = model.train(steps=20)
assert len(history) == 20 and np.isfinite(history).all()
print("Training losses:", history)

## Read the results

A three-qubit output has eight probabilities in computational-basis order. The finite sample can differ from the exact probabilities. Compare both with the two occupied target bins.

In [ ]:
probabilities = model.get_probs()
bits = model.sample(shots=32, return_bits=True)
final_loss = float(model.loss().detach())
print("Probabilities:", probabilities.detach().cpu().numpy())
print("Sample shape:", bits.shape)
print("Post-training loss:", final_loss)
assert probabilities.shape == (2**n_qubits,)
assert bits.shape == (32, n_qubits)

In [ ]:
bins = np.arange(8)
target = np.array([0.5, 0, 0, 0, 0, 0, 0, 0.5])
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(bins-0.18, initial_probs.cpu().numpy(), width=0.36, label="Before")
ax.bar(bins+0.18, probabilities.detach().cpu().numpy(), width=0.36, label="After")
ax.scatter(bins, target, marker="x", color="black", label="Target")
ax.set(ylabel="Probability", xticks=bins, xticklabels=[f"{i:03b}" for i in bins])
ax.legend()
plt.show()

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
fig, ax = draw_qcbm(model)
display(fig)
plt.close(fig)

### Preview the paper figure

Request standalone TikZ for a vector PDF. Symbolic labels describe the circuit architecture. Set `symbolic_parameters=False` to inspect the gate angles at this input.

The helper uses temporary build files. To keep the displayed source, explicitly run `Path("circuit.tex").write_text(tex, encoding="utf-8")`, or call `builder.save_tikz(...)` when working with a builder.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
import subprocess
from IPython.display import Image, display


def show_tikz(source, name="Quantum circuit"):
    """Preview standalone TikZ inline. The optional name is the image description."""
    gs = shutil.which("gs") or shutil.which("gswin64c")
    if not shutil.which("pdflatex") or not gs:
        print("TikZ preview requires pdfLaTeX with Quantikz and Ghostscript.")
        return

    with TemporaryDirectory() as tmp:
        (Path(tmp) / "circuit.tex").write_text(source, encoding="utf-8")
        latex = subprocess.run(
            ["pdflatex", "-no-shell-escape", "-interaction=nonstopmode",
             "-halt-on-error", "circuit.tex"], cwd=tmp, capture_output=True, text=True,
        )
        if latex.returncode:
            raise RuntimeError(latex.stdout[-2000:])
        png = subprocess.check_output(
            [gs, "-q", "-dSAFER", "-sDEVICE=png16m", "-r120", "-o", "-", "circuit.pdf"],
            cwd=tmp,
        )
    display(Image(data=png, alt=name))

In [ ]:
tex = draw_qcbm(
    model, output="tikz",
    tikz_options={"standalone": True, "symbolic_parameters": True, "row_sep": "0.7cm"},
)
show_tikz(tex, "generative")

## Try it

Change the bandwidths or extend training. Compare mass on 000 and 111 with the initial mass and check whether both modes are represented.